# Semana 17 — Instruction Dataset Engineering (JSONL)
**Fase 4: Parameter-Efficient Fine-Tuning (PEFT)** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/notebooks/semana_17_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-17-Conceitos.md](https://github.com/NetoRibeiro/applied-ai-architecture/blob/main/04-Phase-4-Fine-Tuning/docs/Semana-17-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_17_gabarito.ipynb`.

In [ ]:
import json, re, random, unicodedata, collections
import pandas as pd, matplotlib.pyplot as plt
from typing import Literal
from pydantic import BaseModel, Field, ValidationError

## Passo 1 — Auditoria dos dados brutos
Chamados de TI fictícios, com a sujeira de dados reais. Explore antes de mexer.

In [ ]:
# --- Dados brutos: chamados de TI fictícios, com a sujeira típica de dados reais ---
import random
SISTEMA_TICKETS = ('Você é um classificador de chamados de TI. Responda apenas com um JSON com as chaves '
                   '"categoria", "prioridade" e "resumo".')
CATEGORIAS = ["acesso", "hardware", "software", "rede", "financeiro"]; PRIORIDADES = ["baixa", "media", "alta"]
def gerar_tickets_brutos(n=600, seed=0):
    rnd = random.Random(seed)
    modelos = {
        "acesso": [("Não consigo entrar no {x}, aparece senha inválida.", "Falha de login no {x}"),
                   ("Meu usuário foi bloqueado no {x} depois de várias tentativas.", "Usuário bloqueado no {x}")],
        "hardware": [("O {x} não liga desde ontem.", "{x} não liga"), ("A tela do {x} está piscando sem parar.", "Tela do {x} com defeito")],
        "software": [("O {x} fecha sozinho quando abro um arquivo grande.", "{x} fecha inesperadamente"),
                     ("Preciso instalar o {x} na minha máquina.", "Instalação do {x}")],
        "rede": [("A internet do {x} está muito lenta.", "Lentidão de rede no {x}"), ("O wifi do {x} cai a cada dez minutos.", "Queda de wifi no {x}")],
        "financeiro": [("Minha nota de reembolso de {x} reais não foi paga.", "Reembolso de {x} reais pendente"),
                       ("Fui cobrado em duplicidade em {x} reais na fatura.", "Cobrança duplicada de {x} reais")],
    }
    slots = {"acesso": ["SAP", "portal RH", "CRM", "e-mail corporativo", "VPN"], "hardware": ["notebook", "monitor", "teclado", "headset", "dock"],
             "software": ["Excel", "Power BI", "Teams", "VS Code", "Outlook"], "rede": ["3º andar", "escritório de Lisboa", "auditório", "sala de reuniões", "home office"],
             "financeiro": ["150", "480", "1200", "75", "320"]}
    urgencia = {"alta": ["Isso está parando toda a equipe!", "É urgente, tenho uma entrega hoje."],
                "media": ["Preciso disso resolvido esta semana.", "Está atrapalhando meu trabalho."],
                "baixa": ["Sem pressa.", "Quando puderem, por favor."]}
    saudacoes = ["Olá,", "Bom dia!", "Oi pessoal,", "Prezados,", ""]
    nomes = ["ana.souza", "joao.lima", "carla.mendes", "pedro.alves", "rita.costa"]
    linhas = []
    for _ in range(n):
        cat = rnd.choice(CATEGORIAS); texto, resumo = rnd.choice(modelos[cat]); x = rnd.choice(slots[cat]); prio = rnd.choice(PRIORIDADES)
        corpo = f"{rnd.choice(saudacoes)} {texto.format(x=x)} {rnd.choice(urgencia[prio])}"
        if rnd.random() < 0.5: corpo += f"  Att, {rnd.choice(nomes)}@empresa.com, tel (11) 9{rnd.randint(1000, 9999)}-{rnd.randint(1000, 9999)}"
        if rnd.random() < 0.2: corpo = "   " + corpo.replace(" ", "  ", 2)
        rot_cat = rnd.choice([cat, cat, cat.upper(), f" {cat} "]); rot_prio = rnd.choice([prio, prio, prio.capitalize(), "média" if prio == "media" else prio])
        linhas.append({"texto": corpo, "categoria": rot_cat, "prioridade": rot_prio, "resumo": resumo.format(x=x)})
    linhas += [dict(rnd.choice(linhas)) for _ in range(n // 12)]                      # duplicatas exatas
    linhas += [{"texto": "", "categoria": "acesso", "prioridade": "baixa", "resumo": ""} for _ in range(n // 40)]   # vazios
    rnd.shuffle(linhas); return linhas
brutos = gerar_tickets_brutos()
df = pd.DataFrame(brutos); print(len(df), "linhas"); df.head(8)

In [ ]:
print("categorias:", sorted(df["categoria"].unique()))
print("prioridades:", sorted(df["prioridade"].unique()))
print("textos vazios:", (df["texto"].str.strip() == "").sum(), "| duplicatas exatas:", df.duplicated("texto").sum())
print("com e-mail:", df["texto"].str.contains("@").sum())

Liste os problemas que você vê. Cada um vira um passo de limpeza.

## Passo 2 — Normalizar o texto e mascarar dados pessoais
`limpar_texto`: reduza sequências de espaços a um só, remova espaços das pontas, troque e-mails por `<EMAIL>` e telefones no formato `(11) 91234-5678` por `<TELEFONE>`.

In [ ]:
def limpar_texto(t):
    # [SEU CÓDIGO AQUI] espaços, e-mail e telefone

assert limpar_texto("  Oi   ana.s@empresa.com  tel (11) 91234-5678 ") == "Oi <EMAIL> tel <TELEFONE>"

## Passo 3 — Normalizar labels
`ACESSO`, ` acesso ` e `acesso` são a mesma classe; `Média`, `média` e `media` também. Minúsculas, sem espaços nas pontas, sem acentos.

In [ ]:
def normalizar_rotulo(s):
    # [SEU CÓDIGO AQUI] strip, lower e remoção de acentos

assert normalizar_rotulo(" Média ") == "media" and normalizar_rotulo("ACESSO") == "acesso"

## Passo 4 — Schema do target com Pydantic
O target é um objeto com `categoria` (uma de cinco), `prioridade` (uma de três) e `resumo` (texto de 3 a 80 caracteres). Qualquer coisa fora disso deve falhar na validação.

In [ ]:
# [SEU CÓDIGO AQUI] defina a classe Chamado(BaseModel)

print(Chamado(categoria="rede", prioridade="alta", resumo="Queda de wifi"))
try: Chamado(categoria="REDE", prioridade="urgente", resumo="x")
except ValidationError as e: print("rejeitado, como esperado:", len(e.errors()), "erros")

## Passo 5 — Limpar, deduplicar e validar
Percorra os brutos: limpe o texto, descarte vazios e duplicatas (pelo texto limpo), normalize os labels e valide com `Chamado`. Conte quantos foram descartados por cada motivo.

In [ ]:
limpos, vistos, descartes = [], set(), collections.Counter()
for l in brutos:
    t = limpar_texto(l["texto"])
    # [SEU CÓDIGO AQUI] descarte vazios e duplicatas, valide o alvo e acrescente {"texto", "alvo"} em `limpos`
print(len(limpos), "exemplos limpos | descartes:", dict(descartes))

## Passo 6 — Formato de chat
Converta cada exemplo em `{"messages": [system, user, assistant]}`. O conteúdo do assistente é o JSON do target (`model_dump_json()`).

In [ ]:
def para_chat(ex):
    # [SEU CÓDIGO AQUI] devolva o dict com a lista "messages"

exemplos = [para_chat(e) for e in limpos]; print(json.dumps(exemplos[0], ensure_ascii=False, indent=1))

## Passo 7 — Divisão sem vazamento
Embaralhe com seed fixa e divida em 80/10/10. Depois confirme que nenhum texto de usuário aparece em mais de um conjunto.

In [ ]:
random.Random(0).shuffle(exemplos)
# [SEU CÓDIGO AQUI] defina treino, val e teste
u = lambda conj: {e["messages"][1]["content"] for e in conj}
assert not (u(treino) & u(teste)) and not (u(treino) & u(val)) and not (u(val) & u(teste))
print(len(treino), len(val), len(teste), "| sem interseção de textos")

**Atenção:** aqui só eliminamos duplicatas exatas. Chamados gerados do mesmo modelo de frase com outro sistema ("SAP" vs. "CRM") ainda estão dos dois lados. Em um projeto real, você agruparia por modelo/cliente. Quantos textos do teste têm um "quase gêmeo" no treino?

In [ ]:
esqueleto = lambda t: re.sub(r"\d+", "N", re.sub(r"<\w+>", "", t)).lower()[:40]
gemeos = sum(esqueleto(t) in {esqueleto(x) for x in u(treino)} for t in u(teste))
print(f"{gemeos} de {len(teste)} textos de teste têm os mesmos 40 caracteres iniciais de algum texto de treino")

## Passo 8 — Estatísticas
Balanceamento das classes no treino e distribuição de comprimento. Sem tokenizer aqui, estimamos tokens como `caracteres / 3,5` (aproximação razoável para português).

In [ ]:
alvos = [json.loads(e["messages"][-1]["content"]) for e in treino]
# [SEU CÓDIGO AQUI] imprima a proporção de cada categoria e de cada prioridade
comp = [sum(len(m["content"]) for m in e["messages"]) / 3.5 for e in treino]
plt.hist(comp, bins=30); plt.xlabel("tokens (estimado)"); plt.show()
print(f"p50={pd.Series(comp).quantile(.5):.0f} p99={pd.Series(comp).quantile(.99):.0f} máx={max(comp):.0f}")

Que `max_length` você usaria no treino?

## Passo 9 — Gravar e reler o JSONL
Uma linha por exemplo, `ensure_ascii=False`. Depois releia `test.jsonl` validando cada linha: três mensagens, papéis na ordem certa e target aceito por `Chamado`.

In [ ]:
def gravar(caminho, conj):
    # [SEU CÓDIGO AQUI] escreva um JSON por linha

for nome, conj in [("train", treino), ("val", val), ("test", teste)]: gravar(f"{nome}.jsonl", conj)

def validar(caminho):
    n = 0
    for linha in open(caminho, encoding="utf-8"):
        m = json.loads(linha)["messages"]
        # [SEU CÓDIGO AQUI] verifique os papéis e valide o alvo com Chamado.model_validate_json
        n += 1
    return n

print("linhas válidas em test.jsonl:", validar("test.jsonl"))

## Passo 10 — Dataset card
Escreva `DATASET_CARD.md` com: origem, tarefa, etapas de tratamento, tamanho de cada conjunto, limitações conhecidas (inclua a dos "quase gêmeos") e versão.

In [ ]:
card = f"""# Dataset: classificação de chamados de TI (v0.1)
- **Tarefa:** texto do chamado -> JSON com categoria, prioridade e resumo.
- **Origem:** dados sintéticos gerados para estudo.
- **Tratamento:** normalização de espaços, mascaramento de e-mail/telefone, normalização de rótulos, deduplicação exata, validação Pydantic.
- **Tamanho:** treino {len(treino)}, validação {len(val)}, teste {len(teste)}.
- **Limitações:** [SEU TEXTO AQUI]
"""
open("DATASET_CARD.md", "w", encoding="utf-8").write(card); print(card)

## Teste Prático de Entrevista — Validação e Deduplicação de Dataset JSONL
Um entrevistador pede: *"Implemente `limpar_dataset_sft(linhas_jsonl)` que recebe linhas JSONL de chat (`{'messages': [{'role': ..., 'content': ...}, ...]}`), descarta registros com JSON inválido, sem exatamente um `user` e um `assistant` não vazios, e deduplica pelo texto do usuário normalizado (`strip().lower()`). Retorne `(validos, descartados)`."* 

In [ ]:
import json


def limpar_dataset_sft(linhas_jsonl):
    """Valida schema de chat SFT e deduplica pelo prompt do usuário."""
    # [SEU CÓDIGO AQUI]
    pass


raw = [
    json.dumps({"messages": [{"role": "user", "content": "Oi"}, {"role": "assistant", "content": "Olá!"}]}),
    json.dumps({"messages": [{"role": "user", "content": " oi "}, {"role": "assistant", "content": "Duplicado"}]}),
    json.dumps({"messages": [{"role": "user", "content": "Sem resposta"}, {"role": "assistant", "content": "   "}]}),
    "{json invalido",
]
valid, dropped = limpar_dataset_sft(raw)
assert len(valid) == 1 and dropped == 3, f"valid={len(valid)}, dropped={dropped}"
print("OK! SFT dataset limpo:", len(valid), "válido(s),", dropped, "descartado(s).")